In [15]:
from pathlib import Path
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
import os
import matplotlib.pyplot as plt
import cv2
import torch.nn as nn
import optuna
import random
import numpy as np

In [14]:
DATASET_DIR = Path("dataset")
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
storage_name = "sqlite:///db/optuna_study4.db"
EPOCHS = 10
out_path =  Path.cwd() / "models" / "model.pt"

Using cpu device


## 2 ) Visualize the dataset

In [ ]:
for class_ in os.listdir(DATASET_DIR / "training"):
    class_path = DATASET_DIR / "training" / class_

    print(f"Class: {class_}")
    # Lista de imágenes de esa clase
    images = os.listdir(class_path)[:5]

    fig = plt.figure(figsize=(20, 5))
    columns, rows = 5, 1

    for i in range(len(images)):
        img_path = class_path / images[i]
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        fig.add_subplot(rows, columns, i + 1)
        plt.imshow(img)
        plt.axis("off")

    plt.show()

## 2 ) Loading the dataset and creating a dataloader

In [16]:
def data_loader(data_path: Path, batch_size , transform : transforms.Compose):
    dataset_transformed = ImageFolder(data_path, transform=transform)
    dataloader = torch.utils.data.DataLoader(dataset_transformed, batch_size=batch_size, shuffle=True)
    return dataloader

## 4 ) CNN Network Architecture : AlexNet

In [17]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=12):
        super(AlexNet, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=96, kernel_size=11, stride=4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(in_channels=96, out_channels=256 , kernel_size=5, stride=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(in_channels=256, out_channels=384, kernel_size=3, stride=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=384, out_channels=384, kernel_size=3, stride=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=384, out_channels=256, kernel_size=3, stride=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2), # Output is [Batch, 256, 6, 6]
            # Allows CNNs to accept images of varying dimensions without changing the network architecture.It automatically calculates the necessary kernel size and stride
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten() # Output is [Batch, 9216]
        )
        self.classifier = nn.Sequential(
            nn.Linear(in_features=256,out_features= 4096),
            nn.ReLU(inplace=True),
            nn.Linear(in_features=4096, out_features= 4096),
            nn.ReLU(inplace=True),
            nn.Linear(in_features=4096, out_features= num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## 5 ) Train

In [18]:
def train(model, train_loader, optimizer, loss_fn, device):
    model.train()
    for data, target in train_loader:
        # Send the data to the device in our case GPU
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()

        # model(data) returns [batch_size, 1].
        # targets are [batch_size], so we remove only the last dimension to match shapes for MSELoss: [batch_size, 1] -> [batch_size] we apply the squeeze function.
        prediction = model(data).squeeze()
        loss = loss_fn(prediction, target)
        loss.backward()

        # Update the model parameters using the computed gradients and the learning rate.
        optimizer.step()

 ## 6 ) Validate

In [19]:
def validation(model, val_loader, loss_fn, device) -> float:
    model.eval()
    total = 0.0
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            prediction = model(data).squeeze()
            total += loss_fn(prediction, target).item()
    return total / len(val_loader)

## 5 ) Optuna Optimization

In [ ]:
def objective(trial):
    loss_fn = nn.CrossEntropyLoss()
    model = AlexNet()

    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128, 256])

    # Hyperparameters to image transformation
    #Image size
    image_size = trial.suggest_categorical("image_size", [224 , 299])

    # Rotation degrees
    rot_degrees = trial.suggest_int("rot_degrees", 0, 360)

    #Probability to flip horizontally
    h_flip = trial.suggest_int("h_flip", 0, 1)

    #Brightness ( 1.0 is the original )
    brightness = trial.suggest_int("brightness", 0.8, 1.2)

    #Randndom crop ( how much are we willing to crop the image randomly )
    crop_scale = trial.suggest_int("crop_scale", 0.7, 1)

    #Find the best normalization transformation
    norm_strategy = trial.suggest_categorical("norm_strategy", ["imagenet", "custom", "simple"])

    if norm_strategy == "imagenet":
        mean = [0.485, 0.456, 0.406]
        std = [0.229, 0.224, 0.225]
    elif norm_strategy == "custom":
        # Optuna search for you mean values and std for our data
        m = trial.suggest_float("m_custom", 0.0, 1.0)
        s = trial.suggest_float("s_custom", 0.1, 0.5)
        mean, std = [m, m, m], [s, s, s]
    else: # "simple" [0, 1]
        mean, std = [0.0, 0.0, 0.0], [1.0, 1.0, 1.0]

    transform = transforms.Compose([
        transforms.RandomRotation(degrees=rot_degrees),
        transforms.RandomResizedCrop(size=(image_size,image_size),scale=(crop_scale,crop_scale,1)),
        transforms.RandomHorizontalFlip(p=h_flip),
        transforms.ColorJitter(brightness=brightness),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    train_loder = data_loader(DATASET_DIR / "training", batch_size=batch_size, transform=transform)
    validation_loader = data_loader(DATASET_DIR / "validation", batch_size=batch_size, transform=transform)

    #Generate the best Optimizer
    #Suggest best lr
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)

    #Get optimizer name
    optimizer = trial.suggest_categorical("optimizer", ["Adam", "SGD"])

    momentum = trial.suggest_float("momentum", 0.8, 1)

    #Penalizes large weights , prevents overfitting
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-1, log=True)

    if optimizer == "Adam":
        scaling = trial.suggest_float("scaling", 0.9, 1)
        optimizer = torch.optim.Adam(
            params= model.parameters(),
            betas=(momentum,scaling),
            lr=lr,
            weight_decay=weight_decay,
        )
    else:
        optimizer = torch.optim.SGD(
            params=model.parameters(),
            momentum= momentum,
            lr=lr,
            weight_decay=weight_decay)

    for _ in range(EPOCHS):
        train(model, train_loder, optimizer, loss_fn, device)

    return validation(model,validation_loader,loss_fn, device)

## 6 ) Run Optuna Optimization

In [ ]:
study = optuna.create_study(
    study_name="study8",
    storage=storage_name,
    direction="maximize",
    load_if_exists=True
)

study.optimize(objective, n_trials=20)

In [20]:
def get_transform(best_params):
    # Si recibe un objeto Optuna study, extrae .best_params
    if hasattr(best_params, "best_params"):
        best_params = best_params.best_params

    image_size = best_params["image_size"]
    rot_degrees = best_params["rot_degrees"]
    h_flip = best_params["h_flip"]
    brightness = best_params["brightness"]
    crop_scale = best_params["crop_scale"]
    norm_strategy = best_params["norm_strategy"]

    if norm_strategy == "imagenet":
        mean = [0.485, 0.456, 0.406]
        std = [0.229, 0.224, 0.225]

    elif norm_strategy == "custom":
        m = best_params["m_custom"]
        s = best_params["s_custom"]
        mean, std = [m, m , m ], [s, s , s]

    else :
        mean, std = [0.0, 0.0, 0.0], [1.0, 1.0, 1.0]

    return transforms.Compose([
            transforms.RandomRotation(degrees=rot_degrees),
            transforms.RandomResizedCrop(size=(image_size,image_size),scale=(crop_scale,crop_scale,1)),
            transforms.RandomHorizontalFlip(p=h_flip),
            transforms.ColorJitter(brightness=brightness),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])

In [21]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            predictions = torch.argmax(output, dim=1)

            # Count correct predictions
            correct += (predictions == target).sum().item()
            total += target.size(0)

    accuracy = correct / total if total > 0 else 0.0
    return accuracy

In [22]:
def save_checkpoint(model, best_params, checkpoint_path="model.pt"):

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "best_params": best_params,
    }

    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved to {checkpoint_path}")

## 7 ) Train the model with the best hyperparameters and save the best model

In [ ]:
study = optuna.load_study(study_name="study7", storage=storage_name)
model = AlexNet().to(device)
loss_fn=nn.CrossEntropyLoss()

lr = study.best_params["lr"]
momentum = study.best_params["momentum"]
weight_decay = study.best_params["weight_decay"]
scaling = study.best_params["scaling"]

optimizer = torch.optim.Adam(
            params= model.parameters(),
            betas=(momentum,scaling),
            lr=lr,
            weight_decay=weight_decay) \
    if study.best_params["optimizer"] == "Adam" else (torch.optim.SGD(
            params= model.parameters(),
            momentum= momentum,
            lr=lr,
))

batch_size = study.best_params["batch_size"]

transform = get_transform(study)
train_loader = data_loader(DATASET_DIR / "training", batch_size=batch_size, transform=transform)

test_loader = data_loader(DATASET_DIR / "test", batch_size=batch_size, transform=transform)

early_stopping = False
best_val_loss = np.inf

    #Training Process
for epoch in range(EPOCHS):
        # Train
        train(model, train_loader, optimizer, loss_fn, device)
        val_loss = validation(model, test_loader, loss_fn,device)
        # Compute accuracy
        train_acc = compute_accuracy(model, train_loader,device)
        val_acc = compute_accuracy(model, test_loader,device)

        print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {val_loss:.6f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stopping = True
            # Save best model
            save_checkpoint(model, study.best_params, checkpoint_path=out_path)

print(f"raining completed!")

if not early_stopping:
    save_checkpoint(model, study.best_params, checkpoint_path=out_path)

print(f"Saved model {out_path}")

Epoch 1/10 - Train Loss: 255897408.000000, Train Acc: 0.0857, Val Acc: 0.0783
Checkpoint saved to C:\Users\beni7\PycharmProjects\Deep-Learning\src\assignment-2\CNN\models\model.pt
Epoch 2/10 - Train Loss: 136.961756, Train Acc: 0.0836, Val Acc: 0.0812
Checkpoint saved to C:\Users\beni7\PycharmProjects\Deep-Learning\src\assignment-2\CNN\models\model.pt
Epoch 3/10 - Train Loss: 5.017704, Train Acc: 0.0877, Val Acc: 0.0841
Checkpoint saved to C:\Users\beni7\PycharmProjects\Deep-Learning\src\assignment-2\CNN\models\model.pt


## 8 ) Evaluate the model on the test set

In [ ]:
def run_evaluation(model_path,num_examples):

    checkpoint = torch.load(model_path, map_location=device)
    model = AlexNet().to(device)

    #  Load the weights
    model.load_state_dict(checkpoint["model_state_dict"])

    transform = get_transform(checkpoint["best_params"])
    test_dataset = ImageFolder(DATASET_DIR / "test", transform=transform)

    indices = random.sample(range(len(test_dataset)), num_examples)
    results = []

    with torch.no_grad():
        for idx in indices:

            original_img, label = test_dataset[idx]
            original_img =original_img.unsqueeze(0).to(device)

            output = model(original_img)
            # Convert logits to probabilities
            probabilities = torch.softmax(output, dim=1)
            prediction = torch.argmax(probabilities, dim=1).item()

            results.append({
                "index": idx,
                "prediction": prediction,
                "actual": label
            })

            print(f"Index: {idx} | Pred: {prediction} | Actual: {label}")

    return results

In [ ]:
results = run_evaluation(out_path, num_examples=10)